# TermoRed S.A. — Capa Silver

Objetivo: convertir el Bronze (todo `string`, con defectos) en un conjunto
de tablas tipadas, normalizadas y confiables, dejando registro de
cada dato inválido.

Ningún registro se descarta en silencio. Todo lo que no
pasa una regla se escribe en `errores_calidad` con la regla que violó.

### Reglas implementadas

| Regla | Descripción | Acción |
|---|---|---|
| D1 | `sensor_id` o `timestamp` nulo | Cuarentena |
| D2 | `sensor_id` inexistente (integridad referencial) | Cuarentena |
| D3 | Temperatura `-999` (falla de sensor) | Cuarentena |
| D4 | Humedad fuera de 0–100 | Se anula el campo, la fila se conserva |
| D5 | Temperatura con coma decimal | Se corrige |
| D6 | Timestamp en `DD/MM/YYYY` | Se corrige |
| D7 | Lectura de sensor dado de BAJA | Cuarentena |
| D8 | Duplicado por (`sensor_id`, `timestamp`) | Se conserva 1, resto a cuarentena |
| D9 | `lectura_id` repetido | Cuarentena |
| D10 | Provincia con formato inconsistente | Se normaliza |
| D11 | `banco_id` duplicado | Se conserva 1, resto a cuarentena |
| D12 | `zona_electrica` nula | Cuarentena |
| D13 | `rango_min`/`rango_max` nulo | Se imputa por `tipo_equipo` |
| D14 | `estado` con formato inconsistente | Se normaliza |
| D15 | `legajo` duplicado | Se conserva 1, resto a cuarentena |
| D16 | `turno` con formato inconsistente | Se normaliza |
| D17 | Corte sin `fecha_hora_fin` | Cuarentena |
| D18 | `duracion_min` inconsistente o nula | Se recalcula |

## 1. Librerías, conexión y rutas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

CUENTA     = "destorageintegration"
CONTENEDOR = "termored-central-us-dev"
CLAVE = dbutils.secrets.get("termored_scope", "storage_key")

spark.conf.set(f"fs.azure.account.key.{CUENTA}.dfs.core.windows.net", CLAVE)

BASE   = f"abfss://{CONTENEDOR}@{CUENTA}.dfs.core.windows.net"
BRONZE = f"{BASE}/1_bronze"
SILVER = f"{BASE}/2_silver"

## 2. Funciones auxiliares

In [0]:
def guardar_parquet(df, destino, nombre, versionar=False):
    temp_path = f"{destino}_tmp"
    (df.coalesce(1).write.mode("overwrite")
       .option("compression", "snappy").parquet(temp_path))
    archivo = [f.path for f in dbutils.fs.ls(temp_path) if f.name.endswith(".parquet")][0]
    sufijo = "_" + datetime.now().strftime("%Y%m%d_%H%M%S") if versionar else ""
    final = f"{destino}/{nombre}{sufijo}.parquet"
    dbutils.fs.mv(archivo, final)
    dbutils.fs.rm(temp_path, recurse=True)
    return final


def a_snake(df):
    """Nombres de columna en snake_case, sin acentos ni espacios."""
    import re
    nuevos = {}
    for c in df.columns:
        n = c.strip()
        n = n.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnN"))
        n = re.sub(r"(?<!^)(?=[A-Z])", "_", n)      # CamelCase -> Camel_Case
        n = re.sub(r"[^0-9a-zA-Z]+", "_", n).strip("_").lower()
        nuevos[c] = n
    for viejo, nuevo in nuevos.items():
        if viejo != nuevo:
            df = df.withColumnRenamed(viejo, nuevo)
    return df


def limpiar_espacios(df):
    """trim sobre todas las columnas de texto; cadena vacía -> NULL."""
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None)
                                   .otherwise(F.trim(F.col(c))))
    return df


def sin_acentos(col):
    return F.translate(col, "áéíóúüÁÉÍÓÚÜñÑ", "aeiouuAEIOUUnN")


PATRON_NUM = r"^-?\d+([.,]\d+)?$"

def a_num(nombre_col):
    """Convierte a double contemplando coma decimal (D5).
    Devuelve NULL si el texto no es numérico, sin lanzar error."""
    v = F.trim(F.col(nombre_col))
    return F.when(v.rlike(PATRON_NUM),
                  F.regexp_replace(v, ",", ".").cast("double"))


def a_ts(nombre_col):
    """Parsea timestamp en los dos formatos presentes en el origen (D6)."""
    v = F.trim(F.col(nombre_col))
    return F.coalesce(
        F.when(v.rlike(r"^\d{4}-\d{2}-\d{2}[ ]\d{2}:\d{2}:\d{2}$"), F.to_timestamp(v, "yyyy-MM-dd HH:mm:ss")),
        F.when(v.rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"),    F.to_timestamp(v, "yyyy-MM-dd'T'HH:mm:ss")),
        F.when(v.rlike(r"^\d{4}-\d{2}-\d{2}[ ]\d{2}:\d{2}$"),       F.to_timestamp(v, "yyyy-MM-dd HH:mm")),
        F.when(v.rlike(r"^\d{2}/\d{2}/\d{4}[ ]\d{2}:\d{2}:\d{2}$"), F.to_timestamp(v, "dd/MM/yyyy HH:mm:ss")),
        F.when(v.rlike(r"^\d{2}/\d{2}/\d{4}[ ]\d{2}:\d{2}$"),       F.to_timestamp(v, "dd/MM/yyyy HH:mm")),
        F.when(v.rlike(r"^\d{2}/\d{2}/\d{4}$"),                     F.to_timestamp(v, "dd/MM/yyyy")),
        F.when(v.rlike(r"^\d{4}-\d{2}-\d{2}$"),                     F.to_timestamp(v, "yyyy-MM-dd")),
    )


def a_bool(nombre_col):
    v = F.upper(F.trim(F.col(nombre_col)))
    return (F.when(v.isin("1", "S", "SI", "TRUE", "T", "V", "Y"), F.lit(True))
             .when(v.isin("0", "N", "NO", "FALSE", "F"),          F.lit(False)))


# Acumulador de la cuarentena
_cuarentena = []

def a_cuarentena(df, fuente, clave, regla, detalle=None):
    """Registra filas rechazadas en el log unificado de calidad."""
    _cuarentena.append(df.select(
        F.lit(fuente).alias("fuente"),
        F.col(clave).cast("string").alias("clave"),
        F.lit(regla).alias("regla"),
        (F.col(detalle).cast("string") if detalle else F.lit(None).cast("string")).alias("detalle"),
        F.current_timestamp().alias("fecha_deteccion"),
    ))

## 3. Lectura de Bronze

In [0]:
b_bancos   = a_snake(spark.read.parquet(f"{BRONZE}/bancos"))
b_sensores = a_snake(spark.read.parquet(f"{BRONZE}/sensores"))
b_personal = a_snake(spark.read.parquet(f"{BRONZE}/personal"))
b_lecturas = a_snake(spark.read.parquet(f"{BRONZE}/lecturas"))
b_cortes   = a_snake(spark.read.parquet(f"{BRONZE}/cortes_energia"))

for n, d in [("bancos", b_bancos), ("sensores", b_sensores), ("personal", b_personal),
             ("lecturas", b_lecturas), ("cortes", b_cortes)]:
    print(f"{n:10s} {d.count():6d} filas")

bancos         21 filas
sensores      100 filas
personal       62 filas
lecturas     2000 filas
cortes         30 filas


## 4. Silver — `dim_bancos`

Reglas D10 (provincia inconsistente), D11 (`banco_id` duplicado),
D12 (`zona_electrica` nula).

In [0]:
bancos = limpiar_espacios(b_bancos)

# D10 — normalización de provincia: sin acentos + capitalización uniforme
bancos = bancos.withColumn("provincia", F.initcap(F.lower(sin_acentos(F.col("provincia")))))

# Tipado
bancos = (bancos
          .withColumn("tiene_grupo_electrogeno", a_bool("tiene_grupo_electrogeno"))
          .withColumn("activo",                  a_bool("activo"))
          .withColumn("fecha_alta",              a_ts("fecha_alta").cast("date")))

# D12 — zona_electrica nula: sin zona no se puede cruzar con cortes de energía
sin_zona = bancos.filter(F.col("zona_electrica").isNull())
a_cuarentena(sin_zona, "bancos", "banco_id", "D12_zona_electrica_nula")
bancos = bancos.filter(F.col("zona_electrica").isNotNull())

# D11 — banco_id duplicado: se conserva la primera ocurrencia
w = Window.partitionBy("banco_id").orderBy(F.col("fecha_alta").asc_nulls_last(), F.col("nombre"))
bancos = bancos.withColumn("_rn", F.row_number().over(w))
a_cuarentena(bancos.filter("_rn > 1"), "bancos", "banco_id", "D11_banco_id_duplicado", "nombre")
dim_bancos = bancos.filter("_rn = 1").drop("_rn")

display(dim_bancos)

banco_id,nombre,ciudad,provincia,zona_electrica,tiene_grupo_electrogeno,fecha_alta,activo,archivo_origen,fecha_carga
BSG-001,Banco de Sangre Ciudad de Buenos Aires,Ciudad de Buenos Aires,Buenos Aires,ZE-01,false,2018-04-13,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-002,Banco de Sangre La Plata,La Plata,Buenos Aires,ZE-02,false,2020-09-30,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-003,Banco de Sangre Mar del Plata,Mar del Plata,Buenos Aires,ZE-03,true,2019-02-24,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-004,Banco de Sangre Bahía Blanca,Bahía Blanca,Buenos Aires,ZE-04,false,2018-12-23,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-005,Banco de Sangre Córdoba,Córdoba,Cordoba,ZE-05,false,2018-05-11,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-006,Banco de Sangre Río Cuarto,Río Cuarto,Cordoba,ZE-06,true,2020-06-14,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-007,Banco de Sangre Villa María,Villa María,Cordoba,ZE-01,true,2018-04-19,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-008,Banco de Sangre Rosario,Rosario,Santa Fe,ZE-02,false,2022-09-15,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-009,Banco de Sangre Santa Fe,Santa Fe,Santa Fe,ZE-03,true,2021-02-13,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z
BSG-010,Banco de Sangre Rafaela,Rafaela,Santa Fe,ZE-04,false,2018-01-27,true,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/bancos.csv,2026-08-12T21:11:16.342638Z


## 5. Silver — `dim_sensores`

Reglas D13 (rangos nulos) y D14 (estado inconsistente). Los rangos son
críticos: definen qué cuenta como excursión de temperatura, así que en lugar
de descartar el sensor se imputa el rango normativo según el tipo de equipo.

In [0]:
sensores = limpiar_espacios(b_sensores)

# D14 — estado a mayúsculas sin acentos: ACTIVO / BAJA / MANTENIMIENTO
sensores = sensores.withColumn("estado", F.upper(sin_acentos(F.col("estado"))))

# tipo_equipo normalizado para poder imputar
sensores = sensores.withColumn("_tipo", F.lower(sin_acentos(F.col("tipo_equipo"))))

sensores = (sensores
            .withColumn("rango_min", a_num("rango_min"))
            .withColumn("rango_max", a_num("rango_max"))
            .withColumn("fecha_instalacion", a_ts("fecha_instalacion").cast("date")))

# D13 — imputación de rangos normativos por tipo de equipo
a_cuarentena(
    sensores.filter(F.col("rango_min").isNull() | F.col("rango_max").isNull()),
    "sensores", "sensor_id", "D13_rango_nulo_imputado", "tipo_equipo")

rango_min_def = (F.when(F.col("_tipo").contains("heladera"), F.lit(2.0))
                  .when(F.col("_tipo").contains("freezer"),  F.lit(-30.0))
                  .when(F.col("_tipo").contains("agitador"), F.lit(20.0)))

rango_max_def = (F.when(F.col("_tipo").contains("heladera"), F.lit(6.0))
                  .when(F.col("_tipo").contains("freezer"),  F.lit(-18.0))
                  .when(F.col("_tipo").contains("agitador"), F.lit(24.0)))

dim_sensores = (sensores
                .withColumn("rango_min", F.coalesce(F.col("rango_min"), rango_min_def))
                .withColumn("rango_max", F.coalesce(F.col("rango_max"), rango_max_def))
                .drop("_tipo"))

# Unicidad de sensor_id
w = Window.partitionBy("sensor_id").orderBy(F.col("fecha_instalacion").asc_nulls_last())
dim_sensores = dim_sensores.withColumn("_rn", F.row_number().over(w))
a_cuarentena(dim_sensores.filter("_rn > 1"), "sensores", "sensor_id", "sensor_id_duplicado")
dim_sensores = dim_sensores.filter("_rn = 1").drop("_rn")

display(dim_sensores)

sensor_id,banco_id,tipo_equipo,modelo,rango_min,rango_max,fecha_instalacion,estado,archivo_origen,fecha_carga
SEN-0001,BSG-001,Heladera Hemocomponentes,Frigolab FL-250,2.0,6.0,2024-06-20,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0002,BSG-001,Heladera Hemocomponentes,HemoCool HC-600,2.0,6.0,2024-03-28,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0003,BSG-001,Freezer Plasma,PlasmaFreeze PF-30,-30.0,-18.0,2024-12-13,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0004,BSG-001,Freezer Plasma,PlasmaFreeze PF-30,-30.0,-18.0,2021-04-04,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0005,BSG-001,Agitador Plaquetas,PlateMix PM-12,20.0,24.0,2022-08-16,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0006,BSG-002,Heladera Hemocomponentes,HemoCool HC-400,2.0,6.0,2022-04-22,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0007,BSG-002,Heladera Hemocomponentes,HemoCool HC-400,2.0,6.0,2023-02-18,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0008,BSG-002,Freezer Plasma,PlasmaFreeze PF-45,-30.0,-18.0,2023-07-18,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0009,BSG-002,Freezer Plasma,Frigolab FZ-800,-30.0,-18.0,2023-01-18,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z
SEN-0010,BSG-002,Agitador Plaquetas,PlateMix PM-12,20.0,24.0,2023-01-29,ACTIVO,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/sensores.csv,2026-08-12T21:11:19.112485Z


## 6. Silver — `dim_personal`

Reglas D15 (legajo duplicado) y D16 (turno inconsistente). El turno importa:
es la dimensión que permite después medir el sesgo de puerta abierta en el
turno noche.

In [0]:
personal = limpiar_espacios(b_personal)

# D16 — turno canónico
t = F.upper(sin_acentos(F.col("turno")))
personal = (personal
            .withColumn("turno",
                        F.when(t.startswith("MAN"), "Mañana")
                         .when(t.startswith("TAR"), "Tarde")
                         .when(t.startswith("NOC"), "Noche"))
            .withColumn("rol", F.initcap(F.lower(F.col("rol"))))
            .withColumn("nombre", F.initcap(F.lower(F.col("nombre"))))
            .withColumn("fecha_alta", a_ts("fecha_alta").cast("date")))

a_cuarentena(personal.filter(F.col("turno").isNull()),
             "personal", "legajo", "D16_turno_no_reconocido")

# D15 — legajo duplicado
w = Window.partitionBy("legajo").orderBy(F.col("fecha_alta").asc_nulls_last())
personal = personal.withColumn("_rn", F.row_number().over(w))
a_cuarentena(personal.filter("_rn > 1"), "personal", "legajo", "D15_legajo_duplicado", "nombre")
dim_personal = personal.filter("_rn = 1").drop("_rn")

display(dim_personal)

legajo,nombre,banco_id,turno,rol,fecha_alta,archivo_origen,fecha_carga
LEG-001,Gustavo Herrera,BSG-001,Mañana,Bioquímico,2019-08-11,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-002,Silvia Fernández,BSG-001,Tarde,Auxiliar De Laboratorio,2022-04-30,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-003,Camila Díaz,BSG-001,Noche,Técnico En Hemoterapia,2019-07-08,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-004,Ariel Medina,BSG-002,Mañana,Técnico En Hemoterapia,2017-10-15,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-005,Carla Torres,BSG-002,Tarde,Auxiliar De Laboratorio,2020-01-12,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-006,Sofía Pereyra,BSG-002,Noche,Técnico En Hemoterapia,2018-02-14,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-007,Silvia Ruiz,BSG-003,Mañana,Técnico En Hemoterapia,2019-12-21,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-008,Fernando Díaz,BSG-003,Tarde,Técnico En Hemoterapia,2022-04-25,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-009,Fernando Gómez,BSG-003,Noche,Técnico En Hemoterapia,2023-06-14,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z
LEG-010,Pablo Sosa,BSG-004,Noche,Bioquímico,2020-06-12,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/personal.csv,2026-08-12T21:11:21.15655Z


## 7. Silver — `cortes_energia` (fuente API)

Reglas D17 (corte sin fin) y D18 (`duracion_min` inconsistente). Un corte sin
hora de fin no permite determinar si una lectura cae dentro de la ventana del
corte, así que no sirve para la atribución de causa y va a cuarentena.

In [0]:
cortes = limpiar_espacios(b_cortes)

cortes = (cortes
          .withColumn("fecha_hora_inicio", a_ts("fecha_hora_inicio"))
          .withColumn("fecha_hora_fin",    a_ts("fecha_hora_fin"))
          .withColumn("duracion_min_orig", a_num("duracion_min"))
          .withColumn("causa", F.initcap(F.lower(sin_acentos(F.col("causa"))))))

# D17 — sin fecha de fin (o sin inicio) no hay ventana temporal utilizable
invalidos = cortes.filter(F.col("fecha_hora_inicio").isNull() | F.col("fecha_hora_fin").isNull())
a_cuarentena(invalidos, "cortes_energia", "corte_id", "D17_corte_sin_fecha_fin", "zona_electrica")
cortes = cortes.filter(F.col("fecha_hora_inicio").isNotNull() & F.col("fecha_hora_fin").isNotNull())

# D18 — duración recalculada a partir de las fechas; se marca la discrepancia
cortes = cortes.withColumn(
    "duracion_min",
    F.round((F.col("fecha_hora_fin").cast("long") - F.col("fecha_hora_inicio").cast("long")) / 60.0, 0))

a_cuarentena(
    cortes.filter(F.col("duracion_min_orig").isNull() |
                  (F.abs(F.col("duracion_min_orig") - F.col("duracion_min")) > 1)),
    "cortes_energia", "corte_id", "D18_duracion_recalculada", "duracion_min_orig")

silver_cortes = cortes.drop("duracion_min_orig")
display(silver_cortes)

corte_id,zona_electrica,fecha_hora_inicio,fecha_hora_fin,duracion_min,causa,archivo_origen,fecha_carga
CE-001,ZE-02,2025-06-05T10:30:00Z,2025-06-05T11:10:00Z,40.0,Mantenimiento Programado,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-002,ZE-03,2025-06-07T16:45:00Z,2025-06-07T20:45:00Z,240.0,Mantenimiento Programado,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-003,ZE-01,2025-06-02T08:15:00Z,2025-06-02T09:45:00Z,90.0,Tormenta Electrica,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-004,ZE-01,2025-06-06T13:30:00Z,2025-06-06T15:30:00Z,120.0,Sobrecarga De Red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-006,ZE-01,2025-06-07T13:00:00Z,2025-06-07T17:00:00Z,240.0,Corte Por Obra En Via Publica,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-007,ZE-06,2025-06-07T23:15:00Z,2025-06-08T01:15:00Z,120.0,Sobrecarga De Red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-008,ZE-01,2025-06-07T10:30:00Z,2025-06-07T11:10:00Z,40.0,Mantenimiento Programado,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-009,ZE-05,2025-06-04T21:45:00Z,2025-06-04T23:45:00Z,120.0,Sobrecarga De Red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-010,ZE-06,2025-06-04T17:15:00Z,2025-06-04T18:25:00Z,70.0,Sobrecarga De Red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z
CE-011,ZE-06,2025-06-05T21:15:00Z,2025-06-05T22:45:00Z,90.0,Sobrecarga De Red,abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/0_raw/cortes_energia.json,2026-08-12T21:11:25.347728Z


## 8. Silver — `fact_lecturas`

Es la tabla de hechos; 
primero se parsea, después se evalúan las reglas en cascada (cada fila queda
etiquetada con **la primera** regla que viola) y recién al final se separa
válidos de cuarentena.

In [0]:
lect = limpiar_espacios(b_lecturas)

# --- 8.1 Parseo (D5 coma decimal, D6 doble formato de fecha)
lect = (lect
        .withColumn("ts",             a_ts("timestamp"))
        .withColumn("temperatura_n",  a_num("temperatura"))
        .withColumn("humedad_n",      a_num("humedad"))
        .withColumn("bateria_pct_n",  a_num("bateria_pct"))
        .withColumn("puerta_abierta_b", a_bool("puerta_abierta")))

# --- 8.2 Detección de duplicados
w_lid = Window.partitionBy("lectura_id").orderBy(F.col("ts").asc_nulls_last())
w_st  = Window.partitionBy("sensor_id", "ts").orderBy(F.col("lectura_id"))

lect = (lect
        .withColumn("_rn_lid", F.when(F.col("lectura_id").isNotNull(), F.row_number().over(w_lid)))
        .withColumn("_rn_st",  F.when(F.col("sensor_id").isNotNull() & F.col("ts").isNotNull(),
                                      F.row_number().over(w_st))))

# --- 8.3 Integridad referencial contra dim_sensores (D2) y estado del sensor (D7)
sens = dim_sensores.select(
    F.col("sensor_id"),
    F.col("banco_id").alias("banco_id_sensor"),
    F.col("tipo_equipo"),
    F.col("estado").alias("estado_sensor"),
    F.col("rango_min"), F.col("rango_max"))

lect = lect.join(F.broadcast(sens), on="sensor_id", how="left")

# --- 8.4 Cascada de reglas
lect = lect.withColumn(
    "regla_violada",
    F.when(F.col("sensor_id").isNull() | F.col("timestamp").isNull(), "D1_clave_nula")
     .when(F.col("ts").isNull(),                                      "D6_timestamp_no_parseable")
     .when(F.col("_rn_lid") > 1,                                      "D9_lectura_id_duplicado")
     .when(F.col("_rn_st")  > 1,                                      "D8_duplicado_sensor_timestamp")
     .when(F.col("estado_sensor").isNull(),                           "D2_sensor_inexistente")
     .when(F.col("estado_sensor") == "BAJA",                          "D7_sensor_dado_de_baja")
     .when(F.col("temperatura_n") == -999,                            "D3_falla_sensor_temp_999")
     .when(F.col("temperatura_n").isNull(),                           "D5_temperatura_no_numerica")
)

for regla in ["D1_clave_nula", "D6_timestamp_no_parseable", "D9_lectura_id_duplicado",
              "D8_duplicado_sensor_timestamp", "D2_sensor_inexistente",
              "D7_sensor_dado_de_baja", "D3_falla_sensor_temp_999",
              "D5_temperatura_no_numerica"]:
    a_cuarentena(lect.filter(F.col("regla_violada") == regla),
                 "lecturas", "lectura_id", regla, "sensor_id")

validas = lect.filter(F.col("regla_violada").isNull())

# --- 8.5 D4: humedad fuera de 0–100. La lectura de temperatura sigue siendo
# válida, así que la fila se conserva y solo se anula el campo humedad.
fuera_hum = validas.filter((F.col("humedad_n") < 0) | (F.col("humedad_n") > 100))
a_cuarentena(fuera_hum, "lecturas", "lectura_id", "D4_humedad_fuera_de_rango", "humedad")

validas = validas.withColumn(
    "humedad",
    F.when((F.col("humedad_n") >= 0) & (F.col("humedad_n") <= 100), F.col("humedad_n")))

# --- 8.6 Enriquecimiento y derivadas
hora = F.hour("ts")

fact_lecturas = (validas
    .withColumn("temperatura", F.round(F.col("temperatura_n"), 2))
    .withColumn("bateria_pct", F.col("bateria_pct_n"))
    .withColumn("puerta_abierta", F.coalesce(F.col("puerta_abierta_b"), F.lit(False)))
    .withColumn("banco_id", F.col("banco_id_sensor"))
    .withColumn("fecha", F.to_date("ts"))
    .withColumn("turno", F.when((hora >= 6)  & (hora < 14), "Mañana")
                          .when((hora >= 14) & (hora < 22), "Tarde")
                          .otherwise("Noche"))
    .withColumn("fuera_de_rango",
                (F.col("temperatura") < F.col("rango_min")) |
                (F.col("temperatura") > F.col("rango_max")))
    .withColumn("desvio_c",
                F.when(F.col("temperatura") < F.col("rango_min"),
                       F.round(F.col("rango_min") - F.col("temperatura"), 2))
                 .when(F.col("temperatura") > F.col("rango_max"),
                       F.round(F.col("temperatura") - F.col("rango_max"), 2))
                 .otherwise(F.lit(0.0)))
    .select("lectura_id", "sensor_id", "banco_id", "tipo_equipo", "ts", "fecha", "turno",
            "temperatura", "humedad", "bateria_pct", "puerta_abierta",
            "rango_min", "rango_max", "fuera_de_rango", "desvio_c")
)

print(f"Lecturas en bronze : {b_lecturas.count()}")
print(f"Lecturas válidas   : {fact_lecturas.count()}")
print(f"Excursiones        : {fact_lecturas.filter('fuera_de_rango').count()}")
display(fact_lecturas.limit(20))

Lecturas en bronze : 2000
Lecturas válidas   : 1812
Excursiones        : 167


lectura_id,sensor_id,banco_id,tipo_equipo,ts,fecha,turno,temperatura,humedad,bateria_pct,puerta_abierta,rango_min,rango_max,fuera_de_rango,desvio_c
LEC-00017,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-02T08:03:00Z,2025-06-02,Mañana,4.1,42.0,96.9,false,2.0,6.0,false,0.0
LEC-00143,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-02T16:09:00Z,2025-06-02,Tarde,3.7,38.7,96.9,false,2.0,6.0,false,0.0
LEC-00214,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-02T23:05:00Z,2025-06-02,Noche,4.2,38.0,96.9,false,2.0,6.0,false,0.0
LEC-00279,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-03T08:00:00Z,2025-06-03,Mañana,3.5,44.5,96.1,false,2.0,6.0,false,0.0
LEC-00390,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-03T16:03:00Z,2025-06-03,Tarde,9.7,34.3,95.8,false,2.0,6.0,true,3.7
LEC-00473,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-03T23:01:00Z,2025-06-03,Noche,3.8,52.8,96.2,false,2.0,6.0,false,0.0
LEC-00604,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-04T08:08:00Z,2025-06-04,Mañana,3.3,43.4,95.0,false,2.0,6.0,false,0.0
LEC-00674,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-04T16:04:00Z,2025-06-04,Tarde,3.8,50.4,94.9,false,2.0,6.0,false,0.0
LEC-00820,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-04T23:12:00Z,2025-06-04,Noche,3.8,40.1,95.2,false,2.0,6.0,false,0.0
LEC-00913,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-05T08:12:00Z,2025-06-05,Mañana,3.4,44.0,95.3,false,2.0,6.0,false,0.0


## 9. Tabla unificada de errores de calidad

In [0]:
from functools import reduce

errores_calidad = reduce(lambda a, b: a.unionByName(b), _cuarentena)
errores_calidad.createOrReplaceTempView("errores_calidad")

display(errores_calidad.groupBy("fuente", "regla").count().orderBy("fuente", "regla"))

fuente,regla,count
bancos,D11_banco_id_duplicado,1
bancos,D12_zona_electrica_nula,1
cortes_energia,D17_corte_sin_fecha_fin,2
cortes_energia,D18_duracion_recalculada,2
lecturas,D1_clave_nula,39
lecturas,D2_sensor_inexistente,29
lecturas,D3_falla_sensor_temp_999,29
lecturas,D4_humedad_fuera_de_rango,8
lecturas,D7_sensor_dado_de_baja,45
lecturas,D8_duplicado_sensor_timestamp,1


### 10. Métricas de resumen

In [0]:
total_bronze = b_lecturas.count()
total_silver = fact_lecturas.count()

resumen = spark.createDataFrame([
    ("Lecturas ingresadas (Bronze)", total_bronze),
    ("Lecturas válidas (Silver)",    total_silver),
    ("Lecturas en cuarentena",       total_bronze - total_silver),
    ("Registros en errores_calidad", errores_calidad.count()),
], ["metrica", "valor"]).withColumn(
    "porcentaje", F.round(F.col("valor") * 100.0 / F.lit(total_bronze), 2))

display(resumen)

metrica,valor,porcentaje
Lecturas ingresadas (Bronze),2000,100.0
Lecturas válidas (Silver),1812,90.6
Lecturas en cuarentena,188,9.4
Registros en errores_calidad,206,10.3


## 11. Persistencia en Silver

In [0]:
tablas = {
    "dim_bancos":      dim_bancos,
    "dim_sensores":    dim_sensores,
    "dim_personal":    dim_personal,
    "cortes_energia":  silver_cortes,
    "fact_lecturas":   fact_lecturas,
    "errores_calidad": errores_calidad,
}

for nombre, df in tablas.items():
    ruta = guardar_parquet(df, f"{SILVER}/{nombre}", nombre)
    print(f"OK  {nombre:16s} {df.count():6d} filas -> {ruta}")

OK  dim_bancos           19 filas -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/2_silver/dim_bancos/dim_bancos.parquet
OK  dim_sensores        100 filas -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/2_silver/dim_sensores/dim_sensores.parquet
OK  dim_personal         60 filas -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/2_silver/dim_personal/dim_personal.parquet
OK  cortes_energia       28 filas -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/2_silver/cortes_energia/cortes_energia.parquet
OK  fact_lecturas      1812 filas -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/2_silver/fact_lecturas/fact_lecturas.parquet
OK  errores_calidad     206 filas -> abfss://termored-central-us-dev@destorageintegration.dfs.core.windows.net/2_silver/errores_calidad/errores_calidad.parquet


## 12. Verificación

In [0]:
for nombre in tablas:
    leido = spark.read.parquet(f"{SILVER}/{nombre}")
    print(f"{nombre:16s} {leido.count():6d} filas | {len(leido.columns)} columnas")

display(spark.read.parquet(f"{SILVER}/fact_lecturas").limit(20))

dim_bancos           19 filas | 10 columnas
dim_sensores        100 filas | 10 columnas
dim_personal         60 filas | 8 columnas
cortes_energia       28 filas | 8 columnas
fact_lecturas      1812 filas | 15 columnas
errores_calidad     206 filas | 5 columnas


lectura_id,sensor_id,banco_id,tipo_equipo,ts,fecha,turno,temperatura,humedad,bateria_pct,puerta_abierta,rango_min,rango_max,fuera_de_rango,desvio_c
LEC-00017,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-02T08:03:00Z,2025-06-02,Mañana,4.1,42.0,96.9,false,2.0,6.0,false,0.0
LEC-00143,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-02T16:09:00Z,2025-06-02,Tarde,3.7,38.7,96.9,false,2.0,6.0,false,0.0
LEC-00214,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-02T23:05:00Z,2025-06-02,Noche,4.2,38.0,96.9,false,2.0,6.0,false,0.0
LEC-00279,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-03T08:00:00Z,2025-06-03,Mañana,3.5,44.5,96.1,false,2.0,6.0,false,0.0
LEC-00390,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-03T16:03:00Z,2025-06-03,Tarde,9.7,34.3,95.8,false,2.0,6.0,true,3.7
LEC-00473,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-03T23:01:00Z,2025-06-03,Noche,3.8,52.8,96.2,false,2.0,6.0,false,0.0
LEC-00604,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-04T08:08:00Z,2025-06-04,Mañana,3.3,43.4,95.0,false,2.0,6.0,false,0.0
LEC-00674,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-04T16:04:00Z,2025-06-04,Tarde,3.8,50.4,94.9,false,2.0,6.0,false,0.0
LEC-00820,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-04T23:12:00Z,2025-06-04,Noche,3.8,40.1,95.2,false,2.0,6.0,false,0.0
LEC-00913,SEN-0001,BSG-001,Heladera Hemocomponentes,2025-06-05T08:12:00Z,2025-06-05,Mañana,3.4,44.0,95.3,false,2.0,6.0,false,0.0
